## Práctica 5: Naïve Bayes
Programa 1

#### Escuela Superior de Cómputo
Ingeniería en IA
<br>
Aprendizaje Automático<br>
**Robles Guzmán Naomi Isabel** <br><br>

<p>Consuelo Varinia García Mendoza</p>
<br>

---
## Especificaciones

### I. Para este programa se utilizará el dataset *iris.csv* y *email.csv*
#### I.a Importamos las librerías necesarias

In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd

#### I.b Cargamos como dataframes los datasets

In [47]:
# Cargar datasets
iris_df = pd.read_csv('iris.csv')
emails_df = pd.read_csv('emails.csv')

# Explorar iris
print("=" * 50)
print("DATASET IRIS | Dimensiones: {iris_df.shape}")
print("=" * 50)
print("DATASET EMAILS | Dimensiones: {emails_df.shape}")
print("=" * 50)

DATASET IRIS | Dimensiones: {iris_df.shape}
DATASET EMAILS | Dimensiones: {emails_df.shape}


#### Dividimos los dataframes en 70% conjunto de entrenamiento y 30% de prueba

In [48]:
# Preparar datos de IRIS
X_iris = iris_df.iloc[:, :-1].values  # Primeras 4 columnas (características)
y_iris = iris_df.iloc[:, -1].values   # Última columna (clase)

# Dividir en entrenamiento (70%) y prueba (30%)
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris, y_iris, 
    test_size=0.3, 
    random_state=0, 
    shuffle=True
)

print(f"Conjunto de entrenamiento IRIS: {X_iris_train.shape}")
print(f"Conjunto de prueba IRIS: {X_iris_test.shape}")

Conjunto de entrenamiento IRIS: (105, 4)
Conjunto de prueba IRIS: (45, 4)


In [49]:
# Preparar datos de EMAILS
# Eliminar columna de ID (primera columna)
X_emails = emails_df.iloc[:, 1:-1].values  # Columnas 2 a 3000 (palabras)
y_emails = emails_df.iloc[:, -1].values    # Última columna (spam/no spam)

# Dividir en entrenamiento (70%) y prueba (30%)
X_emails_train, X_emails_test, y_emails_train, y_emails_test = train_test_split(
    X_emails, y_emails, 
    test_size=0.3, 
    random_state=0, 
    shuffle=True
)

print(f"Conjunto de entrenamiento EMAILS: {X_emails_train.shape}")
print(f"Conjunto de prueba EMAILS: {X_emails_test.shape}")

Conjunto de entrenamiento EMAILS: (3620, 3000)
Conjunto de prueba EMAILS: (1552, 3000)


### II. Con el 70% de entrenamiento genera conjuntos de validación con el método de validación cruzada para k=3

In [50]:
# Realiza validación cruzada con k=3 para ambos clasificadores

def validacion_cruzada_naive_bayes(X_train, y_train, dataset_name):
    # Configurar KFold
    kfold = KFold(n_splits=3, shuffle=False)
    
    # Diccionario para almacenar resultados
    resultados = {
        'Normal': [],
        'Multinomial': []
    }
    
    print(f"{'='*60}")
    print(f"VALIDACIÓN CRUZADA - {dataset_name}")
    print(f"{'='*60}")
    
    # Validación cruzada para GaussianNB (Normal)
    print("******** GaussianNB (Distribución Normal) ******** :")
    pliegue = 1
    for i_train, i_val in kfold.split(X_train):
        X_train_fold = X_train[i_train]
        y_train_fold = y_train[i_train]
        X_val_fold = X_train[i_val]
        y_val_fold = y_train[i_val]
        
        # Entrenar y predecir
        modelo_gauss = GaussianNB()
        modelo_gauss.fit(X_train_fold, y_train_fold)
        y_pred = modelo_gauss.predict(X_val_fold)
        
        # Calcular accuracy
        acc = accuracy_score(y_val_fold, y_pred)
        resultados['Normal'].append(acc)
        print(f"   Pliegue {pliegue}: Accuracy = {acc:.4f}")
        pliegue += 1
    
    promedio_normal = np.mean(resultados['Normal'])
    print(f"-> Promedio: {promedio_normal:.4f}")
    
    # Validación cruzada para MultinomialNB
    print(" ********  MultinomialNB (Distribución Multinomial) ******** ")
    pliegue = 1
    for i_train, i_val in kfold.split(X_train):
        X_train_fold = X_train[i_train]
        y_train_fold = y_train[i_train]
        X_val_fold = X_train[i_val]
        y_val_fold = y_train[i_val]
        
        # Entrenar y predecir
        modelo_multi = MultinomialNB()
        modelo_multi.fit(X_train_fold, y_train_fold)
        y_pred = modelo_multi.predict(X_val_fold)
        
        # Calcular accuracy
        acc = accuracy_score(y_val_fold, y_pred)
        resultados['Multinomial'].append(acc)
        print(f"   Pliegue {pliegue}: Accuracy = {acc:.4f}")
        pliegue += 1
    
    promedio_multi = np.mean(resultados['Multinomial'])
    print(f"-> Promedio: {promedio_multi:.4f}")
    
    return resultados, promedio_normal, promedio_multi

Aplicamos la función

In [51]:
# Aplicar validación cruzada a IRIS
resultados_iris, prom_normal_iris, prom_multi_iris = validacion_cruzada_naive_bayes(
    X_iris_train, y_iris_train, "IRIS"
)

# Aplicar validación cruzada a EMAILS
resultados_emails, prom_normal_emails, prom_multi_emails = validacion_cruzada_naive_bayes(
    X_emails_train, y_emails_train, "EMAILS"
)

VALIDACIÓN CRUZADA - IRIS
******** GaussianNB (Distribución Normal) ******** :
   Pliegue 1: Accuracy = 0.9143
   Pliegue 2: Accuracy = 1.0000
   Pliegue 3: Accuracy = 0.9429
-> Promedio: 0.9524
 ********  MultinomialNB (Distribución Multinomial) ******** 
   Pliegue 1: Accuracy = 0.6000
   Pliegue 2: Accuracy = 0.9429
   Pliegue 3: Accuracy = 0.6286
-> Promedio: 0.7238
VALIDACIÓN CRUZADA - EMAILS
******** GaussianNB (Distribución Normal) ******** :
   Pliegue 1: Accuracy = 0.9478
   Pliegue 2: Accuracy = 0.9428
   Pliegue 3: Accuracy = 0.9494
-> Promedio: 0.9467
 ********  MultinomialNB (Distribución Multinomial) ******** 
   Pliegue 1: Accuracy = 0.9453
   Pliegue 2: Accuracy = 0.9379
   Pliegue 3: Accuracy = 0.9469
-> Promedio: 0.9434


#### TABLA 1: RESULTADOS DE LA VALIDACIÓN CRUZADA CON 3 PLIEGUES

In [52]:
print("\nTABLA 1: RESULTADOS DE LA VALIDACIÓN CRUZADA CON 3 PLIEGUES")

tabla1_data = {
    'Dataset': ['iris.csv']*8 + ['emails.csv']*8,
    'Distribución': ['Normal']*4 + ['Multinomial']*4 + ['Normal']*4 + ['Multinomial']*4,
    'Pliegue': ['1', '2', '3', 'Promedio']*4,
    'Accuracy': (
        resultados_iris['Normal'] + [prom_normal_iris] +
        resultados_iris['Multinomial'] + [prom_multi_iris] +
        resultados_emails['Normal'] + [prom_normal_emails] +
        resultados_emails['Multinomial'] + [prom_multi_emails]
    )
}

tabla1 = pd.DataFrame(tabla1_data)

# Función para resaltar los promedios
def resaltar_promedio(row):
    if row['Pliegue'] == 'Promedio':
        return ['background-color: #ffffcc; font-weight: bold'] * len(row)
    return [''] * len(row)

# Formatear y mostrar
display(tabla1.style
        .format({'Accuracy': '{:.4f}'})
        .apply(resaltar_promedio, axis=1)
        .set_properties(**{'text-align': 'center'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#2196F3'), 
                                          ('color', 'white'), 
                                          ('font-weight', 'bold'),
                                          ('text-align', 'center'),
                                          ('font-size', '14px')]},
            {'selector': 'td', 'props': [('text-align', 'center'),
                                          ('padding', '8px')]}
        ])
        .hide(axis='index'))

# Identificar mejores configuraciones
print("\n ========== MEJORES CONFIGURACIONES  ========== ")
mejor_iris = 'Gaussiana' if prom_normal_iris > prom_multi_iris else 'Multinomial'
mejor_emails = 'Gaussiana' if prom_normal_emails > prom_multi_emails else 'Multinomial'

print(f"   IRIS: {mejor_iris} (Accuracy promedio: {max(prom_normal_iris, prom_multi_iris):.4f})")
print(f"   EMAILS: {mejor_emails} (Accuracy promedio: {max(prom_normal_emails, prom_multi_emails):.4f})")


TABLA 1: RESULTADOS DE LA VALIDACIÓN CRUZADA CON 3 PLIEGUES


Dataset,Distribución,Pliegue,Accuracy
iris.csv,Normal,1,0.9143
iris.csv,Normal,2,1.0000
iris.csv,Normal,3,0.9429
iris.csv,Normal,Promedio,0.9524
iris.csv,Multinomial,1,0.6000
iris.csv,Multinomial,2,0.9429
iris.csv,Multinomial,3,0.6286
iris.csv,Multinomial,Promedio,0.7238
emails.csv,Normal,1,0.9478
emails.csv,Normal,2,0.9428



 ========== MEJORES CONFIGURACIONES  ========== 
   IRIS: Gaussiana (Accuracy promedio: 0.9524)
   EMAILS: Gaussiana (Accuracy promedio: 0.9467)


In [57]:
def pruebas_finales(X_train, y_train, X_test, y_test, mejor_modelo_nombre):
    print(f"Modelo seleccionado: {mejor_modelo_nombre}")
    
    # Seleccionar el modelo apropiado
    if mejor_modelo_nombre == 'Gaussiana':
        modelo_final = GaussianNB()
    else:
        modelo_final = MultinomialNB()
    
    # Entrenar con TODO el conjunto de entrenamiento
    modelo_final.fit(X_train, y_train)
    
    # Predecir en el conjunto de prueba
    y_pred = modelo_final.predict(X_test)
    accuracy_final = accuracy_score(y_test, y_pred)
    # Generar reporte de clasificación
    reporte = classification_report(y_test, y_pred)
    print(reporte)

    cm = confusion_matrix(y_test, y_pred)
    
    return  reporte, modelo_final,cm, accuracy_score

# Pruebas finales para IRIS
print("\n" + "═"*70)
print("🌸 DATASET: IRIS")
print("═"*70)
reporte_iris, modelo_iris, cm_iris, accuracy_iris = pruebas_finales(
    X_iris_train, y_iris_train, 
    X_iris_test, y_iris_test,
    mejor_iris
)

print("\n" + "═"*70)
print("📧 DATASET: EMAILS")
print("═"*70)

reporte_emails, modelo_emails, cm_emails, accuracy_emails = pruebas_finales(
    X_emails_train, y_emails_train,
    X_emails_test, y_emails_test,
    mejor_emails
)





══════════════════════════════════════════════════════════════════════
🌸 DATASET: IRIS
══════════════════════════════════════════════════════════════════════
Modelo seleccionado: Gaussiana
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        16
Iris-versicolor       1.00      1.00      1.00        18
 Iris-virginica       1.00      1.00      1.00        11

       accuracy                           1.00        45
      macro avg       1.00      1.00      1.00        45
   weighted avg       1.00      1.00      1.00        45


══════════════════════════════════════════════════════════════════════
📧 DATASET: EMAILS
══════════════════════════════════════════════════════════════════════
Modelo seleccionado: Gaussiana
              precision    recall  f1-score   support

           0       0.98      0.95      0.96      1111
           1       0.88      0.95      0.91       441

    accuracy                           0.95      155

#### TABLA 2: RESULTADOS DE LAS PRUEBAS FINALES

In [64]:
print("📊 TABLA 2: RESULTADOS DE LAS PRUEBAS FINALES")
print("═"*70)

# Crear DataFrame para Tabla 2
tabla2_data = {
    'Dataset': ['iris.csv', 'emails.csv'],
    'Distribución': [mejor_iris, mejor_emails],
    'Accuracy': [accuracy_iris, accuracy_emails]
}

tabla2 = pd.DataFrame(tabla2_data)

# Función para aplicar estilo
def estilo_tabla2(df):
    return df.style\
        .format({'Accuracy': '{:.6f}'})\
        .set_properties(**{
            'text-align': 'center',
            'padding': '10px',
            'font-size': '12px'
        })\
        .set_table_styles([
            {'selector': 'th', 
             'props': [
                 ('background-color', '#FF9800'),
                 ('color', 'white'),
                 ('font-weight', 'bold'),
                 ('text-align', 'center'),
                 ('font-size', '14px'),
                 ('padding', '12px')
             ]},
            {'selector': 'td',
             'props': [
                 ('text-align', 'center'),
                 ('padding', '10px'),
                 ('border', '1px solid #ddd')
             ]},
            {'selector': 'table',
             'props': [
                 ('border-collapse', 'collapse'),
                 ('width', '100%'),
                 ('margin', '20px 0')
             ]}
        ])\
        .hide(axis='index')


display(estilo_tabla2(tabla2))

📊 TABLA 2: RESULTADOS DE LAS PRUEBAS FINALES
══════════════════════════════════════════════════════════════════════


TypeError: unsupported format string passed to function.__format__

In [55]:

def visualizar_matriz_confusion(cm, dataset_name, modelo_nombre, labels=None):

    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Elegir color según dataset
    color = 'Blues' if dataset_name == 'IRIS' else 'Greens'
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap=color, values_format='d', colorbar=True)
    
    plt.title(f'Matriz de Confusión - {dataset_name}\n{modelo_nombre}', 
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Predicción', fontsize=12, fontweight='bold')
    plt.ylabel('Valor Real', fontsize=12, fontweight='bold')
    
    # Ajustar layout
    plt.tight_layout()
    
    # Guardar
    nombre_archivo = f'matriz_confusion_{dataset_name.lower()}_{modelo_nombre.lower()}.png'
    plt.savefig(nombre_archivo, dpi=300, bbox_inches='tight')
    print(f"\n✅ Matriz de confusión guardada: {nombre_archivo}")
    
    # Mostrar
    plt.show()
    plt.close()


In [56]:

# ============================================================================
# RESUMEN FINAL
# ============================================================================

print("\n" + "═"*70)
print("✨ RESUMEN FINAL DE LA PRÁCTICA")
print("═"*70)

print(f"""
📊 IRIS Dataset:
   • Mejor modelo: {mejor_iris} (GaussianNB) 
   • Accuracy validación cruzada: {prom_normal_iris:.4f}
   • Accuracy prueba final: {accuracy_iris:.6f} ({accuracy_iris*100:.2f}%)
   
📧 EMAILS Dataset:
   • Mejor modelo: {mejor_emails} (MultinomialNB)
   • Accuracy validación cruzada: {max(prom_normal_emails, prom_multi_emails):.4f}
   • Accuracy prueba final: {accuracy_emails:.6f} ({accuracy_emails*100:.2f}%)

📁 Archivos generados:
   ✓ matriz_confusion_iris_{mejor_iris.lower()}.png
   ✓ matriz_confusion_emails_{mejor_emails.lower()}.png
   ✓ Tabla 1 (Validación Cruzada)
   ✓ Tabla 2 (Pruebas Finales)
   ✓ Reportes de clasificación completos

✅ ¡Práctica completada exitosamente!
""")

print("═"*70)


══════════════════════════════════════════════════════════════════════
✨ RESUMEN FINAL DE LA PRÁCTICA
══════════════════════════════════════════════════════════════════════


TypeError: unsupported format string passed to function.__format__